# APGI Quick-Start

This notebook walks through the core APGI equations, runs a short trial sequence,
and visualises the global integration signal $S_t$ alongside the adaptive threshold $\theta_t$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import apgi
from apgi.integration import APGICoreIntegration

print(f"apgi version: {apgi.__version__}")

## 1 — Core equations

The three fundamental quantities are:

| Symbol | Formula | Meaning |
|--------|---------|----------|
| $\Pi^i_{\text{eff}}$ | $\Pi^i_{\text{baseline}} \cdot e^{\beta_{SM} \hat{M}}$ | Somatic-marker-gated interoceptive precision |
| $S_t$ | $\Pi^e \|z^e\| + \Pi^i_{\text{eff}} \|z^i\|$ | Global integration signal |
| $\theta_t$ | $\theta_0 + (\kappa_{meta} C - \Delta_{info} I) / \lambda_\theta$ | Adaptive ignition threshold (equilibrium) |


In [ ]:
pi_i_eff = apgi.compute_pi_i_eff(pi_i_baseline=1.0, beta_sm=0.6, M_hat=0.5)
print(f"Pi_i_eff at beta_sm=0.6, M_hat=0.5: {pi_i_eff:.4f}")

S_t = apgi.compute_S_t(pi_e=1.2, z_e=0.8, pi_i_eff=pi_i_eff, z_i=0.5)
theta_t = apgi.theta_equilibrium(C=1.0, I_val=0.5)
print(
    f"S_t = {S_t:.4f}   theta_t = {theta_t:.4f}   ignition = {apgi.ignition_criterion(S_t, theta_t)}"
)


## 1b — Protocol specification v1.1 parameter registry

All eight protocol JSONs (release `1.1.0`) share one reduced empirical model. Its registered ranges are checked
here against the library's simulation defaults. κ, the ATP/bit energy–information bridge, is **uncalibrated**
(`null`) in v1.1 and never sets ignition steepness. Unregistered coefficients get no legacy default.

In [ ]:
import json
from pathlib import Path

from apgi.core import BETA_SM_DEFAULT, GAMMA_SIG_DEFAULT, THETA_0_DEFAULT


def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(filename)


spec = load_protocol("protocol_1_cardiac_eeg.json")["apgi_parameters"]
print("Reduced quantities:", ", ".join(spec["reduced_quantities"]))
print("Threshold ODE:     ", spec["threshold"])
print("Somatic modulation:", spec["somatic_modulation"])
print(f"κ = {spec['kappa']['value']} ({spec['kappa']['status']}, {spec['kappa']['unit']})")

checks = [
    ("θ₀", THETA_0_DEFAULT, spec["theta_0"]["registered_band"]),
    ("γ_sig", GAMMA_SIG_DEFAULT, spec["gamma_sig"]["range"]),
    ("β_SM", BETA_SM_DEFAULT, spec["beta_SM"]["typical_range"]),
]
for name, value, (lo, hi) in checks:
    print(f"  {name:<6} default {value:<5} registered [{lo}, {hi}] → {'inside' if lo <= value <= hi else 'OUTSIDE'}")


## 2 — Running a session with `APGICoreIntegration`

`APGICoreIntegration` manages the rolling $\theta_t$ update and accumulates trial records.

In [ ]:
rng = np.random.default_rng(0)
n = 200

pi_e = rng.uniform(0.8, 1.5, n)
z_e = rng.uniform(0.2, 1.0, n)
pi_i = rng.uniform(0.5, 1.5, n)
z_i = rng.uniform(0.1, 0.8, n)
C_metabolic = rng.uniform(0.5, 2.0, n)
V_information = rng.uniform(0.1, 1.0, n)

integ = APGICoreIntegration(beta_sm=0.6, gamma_sig=5.0)
records = integ.run_sequence(pi_e, z_e, pi_i, z_i, C_metabolic, V_information)

print(f"Ignition rate: {integ.ignition_rate():.3f}")
print(f"Mean S_t:      {integ.mean_S_t():.3f}")
print(f"Mean theta_t:  {integ.mean_theta():.3f}")


## 3 — Visualising ignition dynamics

In [ ]:
S_series = np.array([r.S_t for r in records])
theta_series = np.array([r.theta_t for r in records])
fired = np.array([r.ignition for r in records])
t = np.arange(n)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, S_series, lw=1.5, color="#2166ac", label=r"$S_t$")
ax.plot(t, theta_series, lw=1.5, color="#d6604d", ls="--", label=r"$\theta_t$")
ax.vlines(t[fired], 0, S_series.max(), colors="#f4a582", alpha=0.4, lw=0.8)
ax.set_xlabel("Trial")
ax.set_ylabel("Signal")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4 — Signal normalization across sessions

`APGINormalizer` fits a z-score or min-max scaler on one session and applies it to another.

In [ ]:
from apgi.normalizer import APGINormalizer

norm = APGINormalizer(method="zscore")
S_normed = norm.fit_transform(S_series)
print(f"After normalisation:  mean={S_normed.mean():.2e}  std={S_normed.std():.4f}")